# Dataset Jaffe
## 

### Load all required packages

In [ ]:
# import all needed R packages
library(ChAMP)
library(ChAMPdata)
library(ggplot2)
library(stringr)
library(ggpubr)
library(RColorBrewer)
library(colorspace)
library(tidyr)
library(tibble)
library(dplyr)
library(IlluminaHumanMethylation450kanno.ilmn12.hg19)
source("/home/jovyan/notebooks/scripts/custom_functions.R")
print("All libraries successfully loaded")

### Set the dataset location

In [ ]:
# set the location of the data directory
# the .idat fluorescence files and the samplesheet.csv have to be located inside the directory and it's subdirectories
getwd()
data_dir <- "/home/jovyan/datasets/Jaffe"
print("Dataset directory successfully set")

### Load the dataset, filter the probes and visualize the data for QC

In [ ]:
# Load in the data
myLoad <- champ.load(directory = data_dir,
                     method="ChAMP", # replaces old loading method using minfi
                     methValue="B", # wether to calculate beta- or M-values
                     autoimpute=TRUE, # if values are missing uses the 3 most similar probes and uses the mean of their values for the missing value 
                     filterDetP=TRUE, # filter single probes, whose methylation signal vs the background signal of the slide is not significant (detPcut)
                     ProbeCutoff=0, # remove all probes with higher missing-value-ratio
                     SampleCutoff=0.1, # remove a full sample if the failed probe ratio (based on p value) is higher
                     detPcut=0.01, # significance p-value cutoff for filterDetP
                     filterBeads=TRUE, # filter out probes if the fraction of samples with a beadcount < 3 is higher than beadCutoff
                     beadCutoff=0.05, # acceptable fraction of samples with a beadcount < 3
                     filterNoCG=TRUE, # wether to remove non-cg probes
                     filterSNPs=TRUE, # wether to remove probes that fallnear a SNP (as defined in Nordlund et al. https://link.springer.com/article/10.1186/s13059-021-02529-2) 
                     population=NULL, # can be assigned to specific population according to www.internationalgenome.org/category/population/
                     filterMultiHit=TRUE, # wether to remove probes that align to multiple genomic locations (also according to Nordlund et al.)
                     filterXY=TRUE, # wether to remove probes on x and y chromosomes
                     force=FALSE, # minfi specific parameter
                     arraytype="450K") # microarray type (can be one of "450K" "EPICv1" or "EPICv2") 

champ.QC(beta = myLoad$beta, # beta values stored in champ.load output
         pheno=myLoad$pd$Sample_Group, # what samplesheet column is your phenotype (where do u expect the major difference between your samples)?
         resultsDir="./CHAMP_QCimages/") # the plots will be saved in the directory this notebook file is located in


In [ ]:
# for two of the normalization methods we need special input, which we generate below
targets <- read.metharray.sheet(data_dir)
rgset <- read.metharray.exp(targets = targets, recursive = TRUE)
mset <- preprocessRaw(rgset)
print("Generation of rgset and mset successfully")

### Search for the best normalization

#### Peak-based correction normalization (PBC)

In [ ]:
# start with PBC normalization
myNormPBC <- champ.norm(beta = myLoad$beta, arraytype = "450K", method = "PBC")
champ.QC(beta = myNormPBC, pheno = myLoad$pd$Sample_Group, resultsDir = "./CHAMP_QCimages/PBC/")

#### Beta-mixture quantile normalization (BMIQ)

In [ ]:
# BMIQ normalization
myNormBMIQ <- champ.norm(beta = myLoad$beta, arraytype = "450K", method = "BMIQ")
champ.QC(beta = myNormBMIQ, pheno=myLoad$pd$Sample_Group, resultsDir="./CHAMP_QCimages/BMIQ/")

#### Functional Normalization (FunNorm)

In [ ]:
myNormFunNorm <- champ.norm(beta = myLoad$beta, arraytype = "450K", method = "FunctionalNormalization", rgSet = rgset)
champ.QC(beta = myNormFunNorm, pheno = myLoad$pd$Sample_Group, resultsDir = "./CHAMP_QCimages/FunNorm/")

#### Question: Which normalization looks best?
Insert the name of the variable that contains the normalization results (myNormPBC, myNormBMIQ, myNormFunNorm) behind the "<-"

In [ ]:
# save best looking normalization result
myNorm <- 

### Generate a SVD plot that highlights significant differences between groups of samples (Significant differences in the group of interest are good)

In [ ]:
champ.SVD(beta=myNorm, pd=myLoad$pd, resultsDir="./CHAMP_SVDimages/")
SVD_custom(beta=myNorm, filename = "SVD_Jaffe.pdf")

### Calculate Differentially Methylated Probes (DMPs) between Samplegroups

In [ ]:
DMPs <- champ.DMP(beta = myNorm,pheno=myLoad$pd$Sample_Group, adjPVal = 0.05, arraytype = "450K")
saveRDS(DMPs, file = "/home/jovyan/myDMP.rds")
saveRDS(myNorm, file = "/home/jovyan/myNorm.rds")
saveRDS(myLoad, file = "/home/jovyan/myLoad.rds")
print("DMP detection completed")

### Visualize and explore the calculated DMPs

In [ ]:
urls <- paste0("https://methylomemasterclass202609.jhaas.gi.denbi.de/user/", Sys.getenv("JUPYTERHUB_USER"), "/shiny/notebooks/App/DMP.GUI/")
cli::cli_text("Click here for interactive exploration of DMPs: {urls}.")

### Calculate Differentially Methylated Regions (DMRs) between Samplegroups

In [ ]:
DMRs <- champ.DMR(beta=myNorm,pheno=myLoad$pd$Sample_Group,method="Bumphunter", arraytype="450K", cores=2)
saveRDS(DMRs, "/home/jovyan/myDMR.rds")
print("DMR detection completed")

### Visualize and explore the calculated DMRs

In [ ]:
urls <- paste0("https://methylomemasterclass202609.jhaas.gi.denbi.de/user/", Sys.getenv("JUPYTERHUB_USER"), "/shiny/notebooks/App/DMR.GUI450K/")
cli::cli_text("Click here for interactive exploration of DMRs: {urls}.")

### Filter the calculated DMRs for significance

In [ ]:
anno <- getAnnotation(IlluminaHumanMethylation450kanno.ilmn12.hg19)

DMRs_filt <- filter_dmrs(DMRs$BumphunterDMR, pval = 0.05, fwer = 0.2,min_abs_value = 0.5)
DMRs_anno <- annotateDMRs(DMRs_filt, anno = anno, myNorm = myNorm)
DMRs_head <- DMRs_anno[1:min(c(nrow(DMRs_anno), 10)), ]
print(DMRs_head)
DMRs_tail <- DMRs_anno[max(c(0,(nrow(DMRs_anno)-9))):nrow(DMRs_anno), ]

### Plot heatmaps for the filtered DMRs
You can use this section in combination with the DMR or DMP GUIs to visualize methylation across samples for genes of interest.

In [ ]:
makeHeatmap_genelist(pdfname = "./Heatmap_images/heatmap_Cortez_Cardoso_genelist.pdf",
                            anno = anno,
                            genelist = c("NPY", "RASSF1", "CLDN10"), # Here you can add gene names of genes that you saw in DMPs/DMRs to see their overall methylation pattern
                                                                     # and find potentially interesting targets for your poster
                            myNorm = myNorm,
                            targets = targets,
                            pd_col = "Sample_Group",
                            compare_values = c("F", "M"))

makeHeatmap_dmr(pdfname = "./Heatmap_images/heatmap_Cortez_Cardoso_dmrs_hypermethylated.pdf",
                dmrs = DMRs_head,
                anno = anno,
                myNorm = myNorm,
                targets = targets,
                pd_col = "Sample_Group",
                compare_values = c("F", "M"))

makeHeatmap_dmr(pdfname = "./Heatmap_images/heatmap_Cortez_Cardoso_dmrs_hypomethylated.pdf",
                dmrs = DMRs_tail,
                anno = anno,
                myNorm = myNorm,
                targets = targets,
                pd_col = "Sample_Group",
                compare_values = c("F", "M"))